# Блок 1. Подготовка данных
**Файл:** Пуски_Бюджет__.xlsx | **Инструмент:** python + pandas

In [7]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

FILE_PATH = r"C:\Users\user\OneDrive\Рабочий стол\Работа\Portfolio projects\НЛМК тестовое\Пуски Бюджет ..xlsx"
df_raw = pd.read_excel(FILE_PATH)
df = df_raw.copy()
print(f"Строк: {len(df)}, Столбцов: {len(df.columns)}")
df.head(3)

Строк: 1615, Столбцов: 14


,Подразделение,СПП проекта,Статус проекта,Фаза проекта,Категория проекта,Наименование проекта,ФИО Менеджера проекта,Программа,Функциональное направление,ПУСК_ПЛАН,ПУСК_ПРОГНОЗ,Утв. Бюдж. Млн руб.,Прог.бюдж.,Изменение от утв. бюджета
0,Проектный офис 1,14-0354,В работе,2,C,Проект 1,Медведев Олег Алексеевич,Поддержание,Производство,NaN,NaN,0.0,0.0,0
1,Проектный офис 4,12-3198,В работе,3,C,Проект 2,Морозов Николай Александрович,Поддержание,Производство,NaN,2024-08-17 00:00:00,351.0,397.0,21
2,Проектный офис 4,14-0359,В работе,2,C,Проект 3,Сидоров Михаил Николаевич,Поддержание,Производство,NaN,NaN,0.0,0.0,0


## Дефект 1. Дубликаты и отсутствующие СПП-коды
Строки без СПП не могут быть однозначно идентифицированы. Дублирующиеся СПП-коды искажают агрегаты.

In [9]:
# Строки без СПП
missing_spp = df[df["СПП проекта"].isna()]
print(f"Строк без СПП: {len(missing_spp)}")
missing_spp[["Наименование проекта", "Статус проекта", "Подразделение"]]

Строк без СПП: 6


,Наименование проекта,Статус проекта,Подразделение
255,Проект 253,Не начат,Проектный офис 4
401,Проект 399,Приостановлен,Проектный офис 1
1200,Проект 1198,Приостановлен,Проектный офис 3
1302,Проект 1300,Приостановлен,Проектный офис 2
1309,Проект 1307,Приостановлен,Проектный офис 3
1613,Проект 1611,В работе,Проектный офис 4


In [10]:
# Дублирующиеся СПП
dup_mask = df["СПП проекта"].duplicated(keep=False) & df["СПП проекта"].notna()
dup_spp = df[dup_mask].sort_values("СПП проекта")
print(f"Строк с дублирующимся СПП: {len(dup_spp)}")
dup_spp[["СПП проекта", "Статус проекта", "ФИО Менеджера проекта"]]

Строк с дублирующимся СПП: 16


,СПП проекта,Статус проекта,ФИО Менеджера проекта
229,22-0726,В архиве,Борисов Руслан Павлович
230,22-0726,В архиве,Попов Андрей Павлович
231,22-0726,В архиве,Медведев Олег Алексеевич
232,22-0726,В архиве,Новиков Сергей Иванович
40,22-0811,В работе,Новиков Сергей Иванович
41,22-0811,Не начат,Киселёв Тимур Владимирович
42,22-0811,В работе,Козлов Павел Михайлович
43,22-0811,В работе,Соколов Роман Сергеевич
44,22-0811,В работе,Степанов Григорий Михайлович
45,22-0811,В работе,Зайцев Евгений Андреевич


In [11]:
# Сохранить проблемные строки для ручной проверки
dup_spp.to_excel("1_duplicates_for_review.xlsx", index=False)
missing_spp.to_excel("1_missing_spp_for_review.xlsx", index=False)

In [12]:
# Очистка: оставить только уникальные ненулевые СПП
df = df[df["СПП проекта"].notna()].drop_duplicates(subset="СПП проекта", keep="first")
print(f"→ После очистки строк: {len(df)}")

→ После очистки строк: 1595


# Дефект 2. Смешанный формат дат (datetime + строка)
Приводим оба столбца к единому типу datetime. Нераспознанные значения → NaT (будут учтены в дефекте 5).

In [14]:
for col in ["ПУСК_ПЛАН", "ПУСК_ПРОГНОЗ"]:
    raw_types = df[col].dropna().apply(type).value_counts()
    str_count = df[col].apply(lambda x: isinstance(x, str)).sum()
    print(f"{col}: {raw_types.to_dict()}, строк с текстовым форматом: {str_count}")

ПУСК_ПЛАН: {<class 'datetime.datetime'>: 709, <class 'str'>: 67}, строк с текстовым форматом: 67
ПУСК_ПРОГНОЗ: {<class 'datetime.datetime'>: 882, <class 'str'>: 67}, строк с текстовым форматом: 67


In [16]:
# Очистка
df["ПУСК_ПЛАН"] = pd.to_datetime(df["ПУСК_ПЛАН"], dayfirst=True, errors="coerce")
df["ПУСК_ПРОГНОЗ"] = pd.to_datetime(df["ПУСК_ПРОГНОЗ"], dayfirst=True, errors="coerce")

# Дефект 3. Опечатка в поле «Статус проекта»
Заменяем 'В архив' → 'В архиве'.

In [17]:
status_counts = df["Статус проекта"].value_counts()
print("Уникальные значения и количество строк:")
print(status_counts.to_string())

Уникальные значения и количество строк:
Статус проекта
В работе         830
В архиве         396
Приостановлен    357
Не начат           9
В архив            2
Завершен           1


In [18]:
# Проверка наличия опечатки
bad_status_count = (df["Статус проекта"] == "В архив").sum()
print(f"Строк с опечаткой 'В архив' (вместо 'В архиве'): {bad_status_count}")

Строк с опечаткой 'В архив' (вместо 'В архиве'): 2


In [20]:
df["Статус проекта"] = df["Статус проекта"].str.strip().replace({"В архив": "В архиве"})
print(f"После замены: {df['Статус проекта'].value_counts().to_string()}")

После замены: Статус проекта
В работе         830
В архиве         398
Приостановлен    357
Не начат           9
Завершен           1


# Дефект 4. Технические даты 1900
Заменяем на NULL.

In [51]:
# Проверка наличия технической даты
tech_plan_date_count = pd.to_datetime(df["ПУСК_ПЛАН"], errors="coerce").eq(pd.Timestamp("1900-01-01")).sum()
print(f"Строк с технической датой в столбце ПУСК_ПЛАН: {tech_plan_date_count}")

tech_forecast_date_count = pd.to_datetime(df["ПУСК_ПРОГНОЗ"], errors="coerce").eq(pd.Timestamp("1900-01-01")).sum()
print(f"Строк с технической датой в столбце ПУСК_ПРОГНОЗ: {tech_forecast_date_count}")

Строк с технической датой в столбце ПУСК_ПЛАН: 0
Строк с технической датой в столбце ПУСК_ПРОГНОЗ: 0


In [52]:
df.loc[df["ПУСК_ПЛАН"] == pd.Timestamp("1900-01-01"), "ПУСК_ПЛАН"] = pd.NaT

df.loc[df["ПУСК_ПРОГНОЗ"] == pd.Timestamp("1900-01-01"), "ПУСК_ПРОГНОЗ"] = pd.NaT

# Дефект 5. Отсутствие дат пуска
При расчете срочных метрик использовать только строки с заполненными датами.

In [53]:
both_missing = df["ПУСК_ПЛАН"].isna() & df["ПУСК_ПРОГНОЗ"].isna()
print(f"Строк без обеих дат пуска: {both_missing.sum()} ({both_missing.mean()*100:.1f}%)")

Строк без обеих дат пуска: 656 (41.1%)


In [54]:
# Разбивка по статусу — где критично
missing_by_status = df[both_missing]["Статус проекта"].value_counts()

print("\nИз них по статусам:")
print(missing_by_status.to_string())


Из них по статусам:
Статус проекта
В работе         273
Приостановлен    188
В архиве         187
Не начат           7
Завершен           1


In [65]:
# Критичный случай: фаза 3-5, статус «В работе», нет прогноза пуска
critical_missing = df[
    df["ПУСК_ПРОГНОЗ"].isna() &
    df["Статус проекта"].isin(["В работе"]) &
    df["Фаза проекта"].isin([3, 4, 5])
]
print(f"\nКритично: активных проектов (фаза 3–5) без прогноза пуска: {len(critical_missing)}")

critical_missing[["СПП проекта", "Наименование проекта", "Фаза проекта",
                   "Подразделение"]].to_excel("5_critical_missing_pusk.xlsx", index=False)
print("→ Файл '5_critical_missing_pusk.xlsx' сохранен для передачи в источник.")


Критично: активных проектов (фаза 3–5) без прогноза пуска: 146
→ Файл '5_critical_missing_pusk.xlsx' сохранен для передачи в источник.


# Дефект 6. Пустое функциональное направление
Заменяем на 'Не указано'.

In [66]:
# Текстовое 'None' (строка) + настоящий NaN
text_none = (df["Функциональное направление"] == "None").sum()
real_nan = df["Функциональное направление"].isna().sum()

print(f"Строк с текстовым 'None': {text_none}")
print(f"Строк с NaN: {real_nan}")
print(f"Итого пустых: {text_none + real_nan}")

Строк с текстовым 'None': 0
Строк с NaN: 0
Итого пустых: 0


In [67]:
df["Функциональное направление"] = df["Функциональное направление"].replace("None", np.nan).fillna("Не указано")

# Дефект 7. Аномалии в бюджете и сроках

In [73]:
# 7a. Активные проекты с нулевым бюджетом (фаза 3+)
zero_budget_active = df[
    df["Статус проекта"].isin(["В работе"]) &
    df["Фаза проекта"].isin([3, 4, 5]) &
    (df["Утв. Бюдж. Млн руб."] == 0)
]

print(f"Активных (фаза 3–5) с нулевым утв. бюджетом: {len(zero_budget_active)}")

Активных (фаза 3–5) с нулевым утв. бюджетом: 0


In [74]:
# 7b. Прогноз пуска раньше плана (логическая ошибка или ускорение — проверить)
df_dates = df[df["ПУСК_ПЛАН"].notna() & df["ПУСК_ПРОГНОЗ"].notna()]
df["Schedule Delay Days"] = (df["ПУСК_ПРОГНОЗ"] - df["ПУСК_ПЛАН"]).dt.days

earlier_than_plan = df[df["Schedule Delay Days"] < -30]  # более 30 дней опережения

print(f"Проектов с прогнозом пуска значительно раньше плана (>30 дней): {len(earlier_than_plan)}")

if len(earlier_than_plan) > 0:
    print(earlier_than_plan[["СПП проекта", "Наименование проекта",
                               "ПУСК_ПЛАН", "ПУСК_ПРОГНОЗ", "Schedule Delay Days"]].head(5).to_string())

Проектов с прогнозом пуска значительно раньше плана (>30 дней): 56
    СПП проекта Наименование проекта  ПУСК_ПЛАН ПУСК_ПРОГНОЗ  Schedule Delay Days
271     12-1846           Проект 269 2023-05-05   2023-01-12               -113.0
314     14-0201           Проект 312 2022-05-31   2022-04-30                -31.0
316     12-2195           Проект 314 2024-12-24   2024-05-02               -236.0
332     12-2130           Проект 330 2022-11-06   2022-08-22                -76.0
341     12-2083           Проект 339 2022-10-27   2022-09-01                -56.0


In [39]:
# 7c. Единственный завершенный проект
completed = df[df["Статус проекта"] == "Завершен"]
print(f"\nПроектов со статусом 'Завершен': {len(completed)} — аномально мало.")


Проектов со статусом 'Завершен': 1 — аномально мало.


In [75]:
print(f"Строк в исходном файле:  {len(df_raw)}")
print(f"Строк после очистки:     {len(df)}")
print(f"Удалено строк:           {len(df_raw) - len(df)} "
      f"(дубли СПП + пустые СПП)")
print()
print("Финальное распределение по статусам:")
print(df["Статус проекта"].value_counts().to_string())
print()
print("Финальное распределение по программам:")
print(df["Программа"].value_counts().to_string())

Строк в исходном файле:  1615
Строк после очистки:     1595
Удалено строк:           20 (дубли СПП + пустые СПП)

Финальное распределение по статусам:
Статус проекта
В работе         830
В архиве         398
Приостановлен    357
Не начат           9
Завершен           1

Финальное распределение по программам:
Программа
Поддержание              1227
Развитие                  243
Поддержание (крупные)     125


In [76]:
# Сохранить очищенный датасет
df.to_excel("data_cleaned.xlsx", index=False)
print("→ Очищенный файл сохранён: 'data_cleaned.xlsx'")

→ Очищенный файл сохранён: 'data_cleaned.xlsx'
